## Data disclosure

This notebook includes selected records from the publicly available CloudFlaws dataset. Because the source data is already public and is used here solely for reproducible security research and tooling demonstration, the sample records have not been redacted or scrubbed.

The data in this notebook is not private, proprietary, customer-derived, employer-derived, or obtained from any non-public environment.

If you want to download for your own project, it can be found here:
https://www.kaggle.com/datasets/nobukim/aws-cloudtrails-dataset-from-flaws-cloud

In [ ]:
# ── Cell 1 (flaws reboot) ── profile the full corpus
import pandas as pd
import numpy as np

df = pd.read_parquet("rawdog/flaws_cloudtrail_logs/flaws_cloudtrail.parquet")   # ~1s, 131MB
print(f"loaded {len(df):,} events  |  {df['eventTime'].min()} → {df['eventTime'].max()}\n")

# ---- where is the volume in time? (monthly) ----
by_month = df.set_index("eventTime").resample("MS").size()
print("═══ events per month (find the active windows) ═══")
peak = by_month.max()
for ts, n in by_month.items():
    bar = "█" * int(40 * n / peak)
    print(f"  {ts:%Y-%m}  {n:>8,}  {bar}")

# ---- principals: who generates the traffic ----
print("\n═══ top 20 principals ═══")
pr = df["principal"].value_counts()
for name, n in pr.head(20).items():
    print(f"  {n:>8,}  {str(name)[:50]}")
print(f"  ... {len(pr)} distinct principals total")

# ---- identity types ----
print("\n═══ id_type ═══")
print(df["id_type"].value_counts(dropna=False).to_string())

# ---- error landscape (78% carry errorCode - what are they?) ----
print("\n═══ top errorCodes ═══")
print(df["errorCode"].value_counts(dropna=False).head(15).to_string())

# ---- read vs write ----
print(f"\nreads (derived): {df['is_read'].mean()*100:.1f}%   writes/actions: {(1-df['is_read'].mean())*100:.1f}%")

# ---- source IP concentration: a few noisy attackers, or broad? ----
print("\n═══ source IP concentration ═══")
ip = df["sourceIP"].value_counts()
print(f"  distinct IPs: {len(ip):,}")
print(f"  top 10 IPs cover {100*ip.head(10).sum()/len(df):.1f}% of all events")
aws_ips = df["sourceIP"].astype(str).str.endswith("amazonaws.com").sum()
print(f"  AWS-service-hostname 'IPs': {aws_ips:,} ({100*aws_ips/len(df):.1f}%)")
print("  top 12 source IPs:")
for ipaddr, n in ip.head(12).items():
    print(f"    {n:>8,}  {ipaddr}")

# ---- quick cross-tab: do high-error principals look different? ----
print("\n═══ per-principal error rate (top 15 by volume) ═══")
g = df.groupby("principal").agg(events=("eventName","size"),
                                err_rate=("errorCode", lambda s: s.notna().mean()),
                                n_ips=("sourceIP","nunique"),
                                n_actions=("eventName","nunique"))
g = g.sort_values("events", ascending=False).head(15)
g["err_rate"] = (g["err_rate"]*100).round(1)
print(g.to_string())

In [ ]:
# ── Cell 2 (flaws reboot) ── per-principal behavioral separation, Aug-2019 excluded
PARQUET = "rawdog/flaws_cloudtrail_logs/flaws_cloudtrail.parquet"   # ~1s, 131MB

import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest

df = pd.read_parquet(PARQUET)

# ---- drop the Aug-2019 runaway spike (one source, 69% of corpus) ----
spike = (df["eventTime"] >= "2019-08-01") & (df["eventTime"] < "2019-09-01")
base = df.loc[~spike].copy()
print(f"corpus {len(df):,} → baseline {len(base):,}  (dropped {spike.sum():,} Aug-2019 events)\n")

# ---- aggregate to per-principal feature vectors ----
def entropy(s):
    p = s.value_counts(normalize=True).values
    return float(-(p * np.log2(p)).sum()) if len(p) else 0.0

def hour_spread(ts):
    hrs = ts.dt.hour
    return hrs.nunique()  # how many distinct hours-of-day active (0-24)

g = base.groupby("principal")
feat = pd.DataFrame({
    "events":       g.size(),
    "err_rate":     g["errorCode"].apply(lambda s: s.notna().mean()),
    "n_ips":        g["sourceIP"].nunique(),
    "n_actions":    g["eventName"].nunique(),
    "n_sources":    g["eventSource"].nunique(),
    "read_ratio":   g["is_read"].mean(),
    "action_entropy": g["eventName"].apply(entropy),
    "hours_active": g["eventTime"].apply(hour_spread),
    "n_regions":    g["awsRegion"].nunique(),
}).fillna(0)

# focus on principals with enough activity to characterize; keep the rest aside
MIN_EVENTS = 50
small = feat[feat["events"] < MIN_EVENTS]
feat = feat[feat["events"] >= MIN_EVENTS].copy()
print(f"principals: {len(feat)} with ≥{MIN_EVENTS} events  ({len(small)} smaller ones set aside)\n")

# ---- model features: log-scale the heavy-tailed counts, keep ratios as-is ----
X = feat.copy()
for c in ["events", "n_ips", "n_actions", "n_sources"]:
    X[c] = np.log1p(X[c])
Xs = StandardScaler().fit_transform(X)

# ---- Isolation Forest at PRINCIPAL granularity (base rate restored) ----
iso = IsolationForest(n_estimators=400, contamination="auto", random_state=42)
feat["if_score"] = iso.fit_predict(Xs)            # -1 outlier / 1 inlier
feat["if_raw"]   = iso.decision_function(Xs)      # lower = weirder

# ---- a simple, explainable composite "weirdness" rank (no model) ----
# z-score the intuitive danger signals and sum; this is the sigwood-style verdict
danger = pd.DataFrame(index=feat.index)
for c in ["err_rate", "n_ips", "n_actions", "action_entropy"]:
    z = (feat[c] - feat[c].mean()) / feat[c].std(ddof=0)
    danger[c] = z
feat["weirdness"] = danger.sum(axis=1)

# ---- show the ranking ----
out = feat.sort_values("weirdness", ascending=False)
show_cols = ["events","err_rate","n_ips","n_actions","action_entropy","hours_active","if_raw","weirdness"]
disp = out[show_cols].copy()
disp["err_rate"] = (disp["err_rate"]*100).round(1)
disp["action_entropy"] = disp["action_entropy"].round(2)
disp["if_raw"] = disp["if_raw"].round(3)
disp["weirdness"] = disp["weirdness"].round(2)
pd.set_option("display.width", 200)
print("═══ principals ranked by composite weirdness (model-free) ═══")
print(disp.head(20).to_string())

# ---- GRADE IT: are the known attackers the standouts? ----
KNOWN_ATTACKERS = {"AIDA9BO36HFBHKGJAO9C1","AIDADO2GQD0K8TEF7KW1V","i-aa2d3b42e5c6e801a"}
KNOWN_BENIGN    = {"secmonkey","cloudsploit_scan","AWSConfig-Describe","AWSService","cloudaux"}

print("\n═══ GRADING (model was never told these labels) ═══")
print("by weirdness rank:")
for rank,(name,row) in enumerate(out.iterrows(),1):
    tag = "🔴 ATTACKER" if name in KNOWN_ATTACKERS else ("🟢 benign" if name in KNOWN_BENIGN else "")
    if name in KNOWN_ATTACKERS or name in KNOWN_BENIGN:
        print(f"  #{rank:>2}  {str(name)[:30]:30s}  weird={row['weirdness']:+.2f}  IF={'OUT' if row['if_score']==-1 else 'in '}  {tag}")

print("\nIsolation Forest flagged as outliers:")
for name,row in feat[feat["if_score"]==-1].sort_values("if_raw").iterrows():
    tag = "🔴" if name in KNOWN_ATTACKERS else ("🟢" if name in KNOWN_BENIGN else "  ")
    print(f"  {tag} {str(name)[:30]:30s}  IF={row['if_raw']:+.3f}  events={int(row['events']):>7,}  err={row['err_rate']*100:.0f}%  ips={int(row['n_ips'])}")

In [ ]:
# ── Cell 3 (flaws reboot) ── per-principal first-seen novelty, Aug-2019 excluded
import pandas as pd
import numpy as np

df = pd.read_parquet(PARQUET)
spike = (df["eventTime"] >= "2019-08-01") & (df["eventTime"] < "2019-09-01")
base = df.loc[~spike].sort_values("eventTime").reset_index(drop=True)
print(f"baseline events (time-ordered): {len(base):,}\n")

# ---- sensitivity lexicon: which actions matter if seen for the first time ----
# verb/keyword based, not an exhaustive SIEM ruleset - sigwood-style low-effort
SENSITIVE_KEYWORDS = (
    "Create","Delete","Put","Attach","Detach","Update","Modify","Add","Remove",
    "Authorize","Revoke","AssumeRole","PassRole","CreateAccessKey","CreateLoginProfile",
    "CreateUser","CreatePolicy","PutUserPolicy","PutRolePolicy","AttachUserPolicy",
    "AttachRolePolicy","CreateKey","Disable","StopLogging","DeleteTrail","Decrypt",
    "GetSecretValue","CreateGrant","ConsoleLogin","CreateInstance","RunInstances",
)
IAM_SOURCES = {"iam","sts","kms","secretsmanager","cloudtrail","organizations"}

def sensitivity(name: str, source: str) -> float:
    s = 1.0
    if any(k in name for k in SENSITIVE_KEYWORDS): s += 2.0
    if source in IAM_SOURCES: s += 1.0
    # writes weigh more than reads (we have is_read derived)
    return s

# corpus-wide action rarity (rarer action → higher novelty weight)
action_freq = base["eventName"].value_counts()
n_events = len(base)
def rarity_weight(name: str) -> float:
    f = action_freq.get(name, 1)
    return np.log10(n_events / f)   # ~0 for ubiquitous, larger for rare

# ---- single pass: detect firsts per principal ----
seen_action = {}   # principal -> set(eventName)
seen_ip     = {}   # principal -> set(sourceIP)
seen_region = {}   # principal -> set(region)
principal_started = {}  # principal -> first eventTime (to know their "age")

firsts = []
for row in base.itertuples(index=False):
    p = row.principal
    sa = seen_action.setdefault(p, set())
    si = seen_ip.setdefault(p, set())
    sr = seen_region.setdefault(p, set())
    if p not in principal_started:
        principal_started[p] = row.eventTime

    new_action = row.eventName not in sa
    new_ip     = row.sourceIP not in si
    new_region = row.awsRegion not in sr

    # only score first-seen ACTIONS as novelty events (ip/region attached as context).
    # skip a principal's very first-ever event (everything is "new" then - not informative)
    if new_action and len(sa) > 0:
        sens = sensitivity(row.eventName or "", row.eventSource or "")
        rar  = rarity_weight(row.eventName or "")
        firsts.append({
            "eventTime": row.eventTime,
            "principal": p,
            "eventName": row.eventName,
            "eventSource": row.eventSource,
            "errored": pd.notna(row.errorCode),
            "new_ip_too": new_ip,
            "new_region_too": new_region,
            "actions_known_before": len(sa),
            "novelty": sens * rar * (0.5 if pd.notna(row.errorCode) else 1.0),  # errored firsts = attempted-but-denied, slightly downweighted
        })
    sa.add(row.eventName); si.add(row.sourceIP); sr.add(row.awsRegion)

F = pd.DataFrame(firsts)
print(f"first-seen action events: {len(F):,} across {F['principal'].nunique()} principals\n")

# ---- top novelty events overall ----
print("═══ top 25 novelty events (first time a principal did a sensitive/rare action) ═══")
top = F.sort_values("novelty", ascending=False).head(25)
print(top[["eventTime","principal","eventName","eventSource","errored","new_ip_too","novelty"]]
      .to_string(index=False))

# ---- THE KEY TEST: novelty per principal, split by whether weirdness already caught them ----
LOUD3 = {"AIDA9BO36HFBHKGJAO9C1","AIDADO2GQD0K8TEF7KW1V","i-aa2d3b42e5c6e801a"}
pnov = F.groupby("principal").agg(
    n_firsts=("eventName","size"),
    max_novelty=("novelty","max"),
    sum_novelty=("novelty","sum"),
    total_events=("principal","size"),
)
# attach how many total events each principal has (to spot the low-and-slow)
tot = base.groupby("principal").size().rename("corpus_events")
pnov = pnov.join(tot)
pnov["caught_by_weirdness"] = pnov.index.isin(LOUD3)

print("\n═══ principals ranked by peak novelty - did we catch anyone NEW? ═══")
rank = pnov.sort_values("max_novelty", ascending=False).head(20)
for name,r in rank.iterrows():
    flag = "←already loud" if r["caught_by_weirdness"] else ("★ LOW-ACTIVITY" if r["corpus_events"] < 50 else "")
    print(f"  {str(name)[:28]:28s} firsts={int(r['n_firsts']):>4}  peak={r['max_novelty']:5.1f}  corpus_evts={int(r['corpus_events']):>7,}  {flag}")

# ---- isolate the complementary win: high-novelty principals that weirdness SET ASIDE (<50 events) ----
print("\n═══ COMPLEMENTARY CATCH: high novelty among the <50-event principals weirdness ignored ═══")
quiet = pnov[(pnov["corpus_events"] < 50) & (~pnov["caught_by_weirdness"])].sort_values("max_novelty", ascending=False).head(10)
if len(quiet):
    print(quiet[["n_firsts","max_novelty","sum_novelty","corpus_events"]].to_string())
    # show what those quiet principals actually did
    for name in quiet.head(5).index:
        acts = F[F["principal"]==name].sort_values("novelty",ascending=False).head(4)
        print(f"\n  {name} (corpus_events={int(pnov.loc[name,'corpus_events'])}):")
        for _,a in acts.iterrows():
            print(f"     {a['eventTime']}  {a['eventName']:28s} {a['eventSource']:14s} err={a['errored']} nov={a['novelty']:.1f}")
else:
    print("  (none - novelty added nothing beyond the loud principals)")

In [ ]:
# ── Cell 4 (flaws reboot) ── lexicon-free novelty: first-seen × corpus-rarity
import pandas as pd
import numpy as np

df = pd.read_parquet(PARQUET)
spike = (df["eventTime"] >= "2019-08-01") & (df["eventTime"] < "2019-09-01")
base = df.loc[~spike].sort_values("eventTime").reset_index(drop=True)

# ---- lane split: MECHANICAL, from userIdentity.type only. no security opinion. ----
# interactive/credentialed principals vs AWS-run service principals.
SERVICE_TYPES = {"AWSService", "AWSAccount"}
def is_service(row):
    if row.id_type in SERVICE_TYPES: return True
    if isinstance(row.invokedBy, str) and row.invokedBy.endswith("amazonaws.com"): return True
    # service-linked role names are an AWS convention, not a judgement call
    if isinstance(row.principal, str) and row.principal.startswith("AWSServiceRoleFor"): return True
    if isinstance(row.principal, str) and row.principal in {"CloudTrail","AWSConfig","configLambdaExecution"}: return True
    return False

base["_service"] = base.apply(is_service, axis=1)
interactive = base[~base["_service"]].copy()
service     = base[base["_service"]].copy()
print(f"baseline {len(base):,}  →  interactive {len(interactive):,} | service {len(service):,}\n")

# ---- corpus rarity: computed over interactive lane, no opinion about which acts matter ----
action_freq = interactive["eventName"].value_counts()
N = len(interactive)
def rarity(name):                     # ~0 for ubiquitous, larger for rare. plain log10 odds.
    return float(np.log10(N / action_freq.get(name, 1)))

# ---- single time-ordered pass: first-seen action per principal ----
seen_action, seen_ip, seen_region, started = {}, {}, {}, {}
firsts = []
for row in interactive.itertuples(index=False):
    p = row.principal
    sa = seen_action.setdefault(p, set())
    si = seen_ip.setdefault(p, set())
    sr = seen_region.setdefault(p, set())
    started.setdefault(p, row.eventTime)
    new_action = row.eventName not in sa
    if new_action and len(sa) > 0:          # skip a principal's very first event (all-new, uninformative)
        firsts.append({
            "eventTime": row.eventTime,
            "principal": p,
            "eventName": row.eventName,
            "eventSource": row.eventSource,
            "errored": pd.notna(row.errorCode),      # COLUMN, not a weight
            "new_ip_too": row.sourceIP not in si,
            "new_region_too": row.awsRegion not in sr,
            "novelty": rarity(row.eventName),         # the ONLY score. first-seen (gate) × rarity (value).
        })
    sa.add(row.eventName); si.add(row.sourceIP); sr.add(row.awsRegion)

F = pd.DataFrame(firsts)
print(f"first-seen action events (interactive lane): {len(F):,} across {F['principal'].nunique()} principals\n")

# ---- top novelty events ----
print("═══ top 25 novelty events - first-seen × rarity, no domain prior ═══")
top = F.sort_values("novelty", ascending=False).head(25)
print(top[["eventTime","principal","eventName","eventSource","errored","new_ip_too","novelty"]].to_string(index=False))

# ---- per-principal novelty rollup ----
LOUD3 = {"AIDA9BO36HFBHKGJAO9C1","AIDADO2GQD0K8TEF7KW1V","i-aa2d3b42e5c6e801a"}
tot = interactive.groupby("principal").size().rename("corpus_events")
pnov = F.groupby("principal").agg(n_firsts=("eventName","size"),
                                  peak=("novelty","max"),
                                  total_novelty=("novelty","sum")).join(tot)
pnov["loud"] = pnov.index.isin(LOUD3)

print("\n═══ principals by total novelty - who does the tool surface? ═══")
for name,r in pnov.sort_values("total_novelty",ascending=False).head(18).iterrows():
    flag = "←already loud in cell 2" if r["loud"] else ("★ quiet (<50 evts)" if r["corpus_events"]<50 else "")
    print(f"  {str(name)[:24]:24s} firsts={int(r['n_firsts']):>4}  peak={r['peak']:.2f}  total={r['total_novelty']:6.1f}  evts={int(r['corpus_events']):>7,}  {flag}")

# ---- the grade: did rarity-alone keep the cell-3 catch? ----
print("\n═══ GRADE: did lexicon-free novelty keep 811596193553? ═══")
if "811596193553" in pnov.index:
    r = pnov.loc["811596193553"]
    rank = pnov.sort_values("total_novelty",ascending=False).index.get_loc("811596193553")+1
    print(f"  811596193553 → rank #{rank} by total novelty, {int(r['n_firsts'])} firsts, peak {r['peak']:.2f}")
    print("  its top first-seen actions (rarity-ranked, no lexicon):")
    for _,a in F[F['principal']=='811596193553'].sort_values('novelty',ascending=False).head(8).iterrows():
        print(f"     {a['eventName']:28s} {a['eventSource']:12s} rarity={a['novelty']:.2f} err={a['errored']}")

# ---- service lane shown separately, not hidden ----
print("\n═══ SERVICE LANE (reported separately, never pooled with interactive) ═══")
svc_first = []
sa2 = {}
for row in service.sort_values("eventTime").itertuples(index=False):
    s = sa2.setdefault(row.principal,set())
    if row.eventName not in s and len(s)>0:
        svc_first.append((row.principal,row.eventName,row.eventSource))
    s.add(row.eventName)
svc_df = pd.DataFrame(svc_first, columns=["principal","eventName","eventSource"])
print(f"  {len(svc_df)} service first-seen events across {svc_df['principal'].nunique() if len(svc_df) else 0} principals (context only)")
if len(svc_df): print("  " + ", ".join(f"{p}:{n}" for p,n,_ in svc_first[:8]))

In [ ]:
# ── Cell 5 (flaws reboot) ── novelty v1: density + burst-aggregation, no priors
import pandas as pd
import numpy as np

df = pd.read_parquet(PARQUET)   # ~1s, 131MB
spike = (df["eventTime"] >= "2019-08-01") & (df["eventTime"] < "2019-09-01")
base = df.loc[~spike].sort_values("eventTime").reset_index(drop=True)

SERVICE_TYPES = {"AWSService","AWSAccount"}
def is_service(row):
    if row.id_type in SERVICE_TYPES: return True
    if isinstance(row.invokedBy,str) and row.invokedBy.endswith("amazonaws.com"): return True
    if isinstance(row.principal,str) and (row.principal.startswith("AWSServiceRoleFor")
        or row.principal in {"CloudTrail","AWSConfig","configLambdaExecution"}): return True
    return False
base["_service"] = base.apply(is_service, axis=1)
interactive = base[~base["_service"]].copy()

action_freq = interactive["eventName"].value_counts()
N = len(interactive)
def rarity(name): return float(np.log10(N / action_freq.get(name,1)))

# ---- pass: collect first-seen actions with rarity + errored ----
seen_action, seen_ip, seen_region = {}, {}, {}
firsts = []
for row in interactive.itertuples(index=False):
    p = row.principal
    sa = seen_action.setdefault(p,set()); si = seen_ip.setdefault(p,set()); sr = seen_region.setdefault(p,set())
    if row.eventName not in sa and len(sa) > 0:
        firsts.append({"eventTime":row.eventTime,"principal":p,"eventName":row.eventName,
                       "eventSource":row.eventSource,"errored":pd.notna(row.errorCode),
                       "rarity":rarity(row.eventName)})
    sa.add(row.eventName); si.add(row.sourceIP); sr.add(row.awsRegion)
F = pd.DataFrame(firsts)

# ════ DENSITY - separate "does rare things" from "does many things" ════
LOUD3 = {"AIDA9BO36HFBHKGJAO9C1","AIDADO2GQD0K8TEF7KW1V","i-aa2d3b42e5c6e801a"}
tot = interactive.groupby("principal").size().rename("corpus_events")
agg = F.groupby("principal").agg(n_firsts=("rarity","size"),
                                 mean_rarity=("rarity","mean"),
                                 median_rarity=("rarity","median"),
                                 err_first_rate=("errored","mean")).join(tot)
# density = share of a principal's activity that is a rare-first. volume-independent.
agg["first_density"] = agg["n_firsts"] / agg["corpus_events"]
# only characterize principals with enough events for density to be stable
agg = agg[agg["corpus_events"] >= 20].copy()
agg["loud"] = agg.index.isin(LOUD3)

print("═══ by NOVELTY DENSITY (rare-firsts per event) - volume-independent ═══")
d = agg.sort_values("first_density", ascending=False).head(18)
for name,r in d.iterrows():
    flag = "←loud(vol)" if r["loud"] else ""
    print(f"  {str(name)[:24]:24s} dens={r['first_density']:.3f}  n_firsts={int(r['n_firsts']):>4}  "
          f"med_rar={r['median_rarity']:.2f}  err_first={r['err_first_rate']*100:4.0f}%  evts={int(r['corpus_events']):>7,}  {flag}")

print("\n═══ same principals, by MEDIAN RARITY of their firsts (how exotic, not how many) ═══")
for name,r in agg.sort_values("median_rarity",ascending=False).head(12).iterrows():
    flag = "←loud(vol)" if r["loud"] else ""
    print(f"  {str(name)[:24]:24s} med_rar={r['median_rarity']:.2f}  dens={r['first_density']:.3f}  evts={int(r['corpus_events']):>7,}  {flag}")

# ════ BURST AGGREGATION - collapse enumeration sweeps to one finding ════
# group a principal's firsts into bursts: consecutive firsts < GAP apart = one burst.
GAP = pd.Timedelta("5min")
F = F.sort_values(["principal","eventTime"])
bursts = []
for p, grp in F.groupby("principal"):
    grp = grp.sort_values("eventTime")
    burst_id = (grp["eventTime"].diff() > GAP).cumsum()
    for bid, b in grp.groupby(burst_id):
        if len(b) >= 3:  # a "burst" = 3+ first-seen actions clustered in time
            bursts.append({"principal":p,"start":b["eventTime"].iloc[0],
                           "span_s":(b["eventTime"].iloc[-1]-b["eventTime"].iloc[0]).total_seconds(),
                           "n_new_actions":len(b),"n_new_sources":b["eventSource"].nunique(),
                           "err_rate":b["errored"].mean(),"mean_rarity":b["rarity"].mean()})
B = pd.DataFrame(bursts)
print(f"\n═══ FIRST-SEEN BURSTS (≥3 new actions within {GAP}) - one row per sweep ═══")
print(f"  {len(B)} bursts across {B['principal'].nunique()} principals\n")
top_b = B.sort_values(["n_new_sources","n_new_actions"], ascending=False).head(15)
for _,r in top_b.iterrows():
    print(f"  {r['start']:%Y-%m-%d %H:%M}  {str(r['principal'])[:22]:22s} "
          f"{int(r['n_new_actions']):>3} new actions / {int(r['n_new_sources']):>2} services "
          f"in {r['span_s']:>5.0f}s  err={r['err_rate']*100:3.0f}%  rar={r['mean_rarity']:.2f}")

# ---- the grade ----
print("\n═══ GRADE ═══")
print("Does density isolate the quiet-unusual from the loud-voluminous?")
dens_rank = agg.sort_values("first_density",ascending=False)
for name in ["811596193553","AIDA9BO36HFBHKGJAO9C1","AIDADO2GQD0K8TEF7KW1V","i-aa2d3b42e5c6e801a","secmonkey","cloudsploit_scan"]:
    if name in dens_rank.index:
        rk = dens_rank.index.get_loc(name)+1
        print(f"  {name[:24]:24s} density rank #{rk:>2}  (vol={int(agg.loc[name,'corpus_events']):>7,})")

In [ ]:
# ── Cell 6 ── Drain3 on a curated projection - flaws OR your own account
import pandas as pd, numpy as np
from drain3 import TemplateMiner
from drain3.template_miner_config import TemplateMinerConfig

# ─── pick your corpus ───
PARQUET = "scratch/cloudtrail_20260520_to_20260603_00h.json.parquet"        # ← or your own account's parquet
DROP_AUG2019 = True                          # set False for your own data

df = pd.read_parquet(PARQUET)
if DROP_AUG2019 and "2019" in str(df["eventTime"].min().year)+str(df["eventTime"].max().year):
    spike = (df["eventTime"]>="2019-08-01") & (df["eventTime"]<"2019-09-01")
    df = df.loc[~spike].copy()
df = df.sort_values("eventTime").reset_index(drop=True)
print(f"{PARQUET}: {len(df):,} events\n")

# ─── curated projection → one text line per event (the syslog-analog) ───
# mechanical lane label, not the specific principal, so templates describe KINDS of activity
SERVICE_TYPES={"AWSService","AWSAccount"}
def lane(r):
    if r.id_type in SERVICE_TYPES: return "svc"
    if isinstance(r.invokedBy,str) and r.invokedBy.endswith("amazonaws.com"): return "svc"
    if isinstance(r.principal,str) and r.principal.startswith("AWSServiceRoleFor"): return "svc"
    return "interactive"
def line(r):
    rw  = "read" if r.is_read else "write"
    err = ("err:"+str(r.errorCode)) if pd.notna(r.errorCode) else "ok"
    return f"{lane(r)} {r.eventSource} {r.eventName} {rw} {err}"

df["_line"] = df.apply(line, axis=1)

# ─── feed lines through Drain3 ───
cfg = TemplateMinerConfig(); cfg.profiling_enabled = False
tm = TemplateMiner(config=cfg)
for ln in df["_line"]:
    tm.add_log_message(ln)

clusters = sorted(tm.drain.clusters, key=lambda c: c.size)
total = len(df)
print(f"Drain3 produced {len(clusters)} templates from {total:,} lines\n")

print("═══ 20 RAREST templates (the surfaced tail) ═══")
for c in clusters[:20]:
    print(f"  n={c.size:>6}  ({100*c.size/total:6.3f}%)  {c.get_template()}")

print("\n═══ 8 most COMMON templates (the baseline being tuned out) ═══")
for c in sorted(clusters, key=lambda c:-c.size)[:8]:
    print(f"  n={c.size:>6}  ({100*c.size/total:6.2f}%)  {c.get_template()}")

# ─── the honest comparison: Drain templates vs a plain value_counts on the same projection ───
vc = df["_line"].value_counts()
print(f"\n═══ COMPARISON: Drain templates={len(clusters)}  vs  distinct raw lines={len(vc)} ═══")
print("rarest 10 raw projections (no Drain, just counting):")
for s,n in vc.tail(10).items():
    print(f"  n={n:>6}  {s}")
print(f"\nIf these two rarest lists look the same, Drain added structure-folding we didn't need on already-structured data.")
print(f"If Drain MERGED lines (templates << distinct lines), it found variable slots worth keeping.")